In [ ]:
#@title Imports
# ============================================
# Cell 1
# Install + import dependencies (Google Colab)
# ============================================

!pip -q install opencv-python-headless pandas numpy

import cv2
import numpy as np
import pandas as pd

print("Environment ready.")

In [ ]:
#@title Contour Parsing
# ============================================
# Cell 2
# Contour (de)serialization
# ============================================


def parse_contour(contour_str):

    if pd.isna(contour_str) or contour_str == "":
        return None

    pts = []

    for p in str(contour_str).split(";"):
        x,y = p.split(":")
        pts.append(
            [
                int(float(x)),
                int(float(y))
            ]
        )

    return np.array(pts,np.int32).reshape(-1,1,2)

In [ ]:
#@title Geometry + Orientation Drawing
# ============================================
# Cell 3
# Geometry reconstruction + per-detection overlay drawing
# ============================================


def compute_axis_box(contour):

    pts = contour.reshape(-1,2).astype(float)

    # -------------------------
    # longest contour distance
    # -------------------------

    max_dist = 0

    for i in range(len(pts)):

        d = np.linalg.norm(
            pts[i+1:] - pts[i],
            axis=1
        )

        if len(d):

            idx=np.argmax(d)

            if d[idx]>max_dist:
                max_dist=d[idx]
                p1=pts[i]
                p2=pts[i+1+idx]

    long_axis=p2-p1
    long_axis/=np.linalg.norm(long_axis)

    short_axis=np.array(
        [
            -long_axis[1],
             long_axis[0]
        ]
    )

    center=pts.mean(axis=0)

    long_proj=np.dot(
        pts-center,
        long_axis
    )

    short_proj=np.dot(
        pts-center,
        short_axis
    )

    lmin,lmax=long_proj.min(),long_proj.max()
    smin,smax=short_proj.min(),short_proj.max()

    corners=np.array(
        [
            center+long_axis*lmin+short_axis*smin,
            center+long_axis*lmax+short_axis*smin,
            center+long_axis*lmax+short_axis*smax,
            center+long_axis*lmin+short_axis*smax
        ],
        dtype=np.int32
    )

    return {
        "center":center,
        "long_axis":long_axis,
        "short_axis":short_axis,
        "lmin":lmin,
        "lmax":lmax,
        "smin":smin,
        "smax":smax,
        "box":corners
    }


def draw_orientation(frame, contour, pose):

    data=compute_axis_box(contour)

    pts=contour.astype(np.int32)

    # contour
    cv2.polylines(
        frame,
        [pts],
        True,
        (0,255,255),
        2
    )

    # bounding box

    box=data["box"]

    cv2.polylines(
        frame,
        [box],
        True,
        (255,0,0),
        2
    )

    # long axis

    c=data["center"]

    p1=c+data["long_axis"]*data["lmin"]
    p2=c+data["long_axis"]*data["lmax"]

    cv2.line(
        frame,
        tuple(p1.astype(int)),
        tuple(p2.astype(int)),
        (0,255,0),
        2
    )

    # midpoint head/tail division line

    mid=(
        data["lmin"]+
        data["lmax"]
    )/2

    mid_point=c+data["long_axis"]*mid

    half_width = (data["smax"] - data["smin"]) / 2

    line1 = mid_point - data["short_axis"]*(half_width*1.5)
    line2 = mid_point + data["short_axis"]*(half_width*1.5)

    cv2.line(
        frame,
        tuple(line1.astype(int)),
        tuple(line2.astype(int)),
        (0,0,255),
        2
    )

    # head/tail regions

    if pose=="ELONGATED":

        pts2=contour.reshape(-1,2)

        long_values=np.dot(
            pts2-c,
            data["long_axis"]
        )

        head_pts=pts2[
            long_values>=mid
        ]

        tail_pts=pts2[
            long_values<mid
        ]

        if len(head_pts):

            hc=head_pts.mean(axis=0)

            cv2.circle(
                frame,
                tuple(hc.astype(int)),
                5,
                (255,0,255),
                -1
            )

        if len(tail_pts):

            tc=tail_pts.mean(axis=0)

            cv2.circle(
                frame,
                tuple(tc.astype(int)),
                5,
                (255,255,0),
                -1
            )

    return frame

In [ ]:
#@title Main Overlay
# ============================================
# Cell 4
# Build annotated overlay video from an analyzed CSV
# (expects the pose/movement/direction_deg columns added by
# multiParams.ipynb's analyze_csv)
# ============================================


def create_overlay(video, csv, output):

    df=pd.read_csv(csv,
                   dtype={"contour_points":str})

    cap=cv2.VideoCapture(video)

    fps=cap.get(
        cv2.CAP_PROP_FPS
    )

    w=int(
        cap.get(cv2.CAP_PROP_FRAME_WIDTH)
    )

    h=int(
        cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
    )

    writer=cv2.VideoWriter(
        output,
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (w,h)
    )

    frame_lookup={}

    for frame_id,g in df.groupby("FRAME"):
        frame_lookup[int(frame_id)] = g

    frame_num=0

    while True:

        ret,frame=cap.read()

        if not ret:
            break

        if frame_num in frame_lookup:

            detections=frame_lookup[frame_num]

            for _,row in detections.iterrows():

                contour=parse_contour(
                    row["contour_points"]
                )

                if contour is None:
                    continue

                frame=draw_orientation(
                    frame,
                    contour,
                    row["pose"]
                )

                x=int(row.POSITION_X)
                y=int(row.POSITION_Y)

                text=(
                    f"ID:{row.TRACK_ID} "
                    f"{row.pose} "
                    f"{row.movement}"
                )

                cv2.putText(
                    frame,
                    text,
                    (x,y-15),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.5,
                    (255,255,255),
                    2
                )

                if not pd.isna(
                    row.direction_deg
                ):

                    cv2.putText(
                        frame,
                        f"{row.direction_deg:.1f} deg",
                        (x,y+15),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        .5,
                        (255,255,255),
                        2
                    )

        writer.write(frame)

        frame_num+=1

    cap.release()
    writer.release()

    print(
        "Saved:",
        output
    )

In [ ]:
# ============================================
# Cell 5
# Notebook inputs + run
# ============================================

# -----------------------------
# Input / output files
# -----------------------------
# VIDEO_PATH : source mp4 (same video the tracks were generated from)
# CSV_PATH   : CSV with pose/movement/direction_deg columns, as produced
#              by multiParams.ipynb's analyze_csv
# OUTPUT_PATH: annotated overlay video to write

VIDEO_PATH  = "/content/C1_20260804_133401.mp4"
CSV_PATH    = "/content/Uranus_cam_test-2_analyzed.csv"
OUTPUT_PATH = "/content/Uranus_cam_test-2_overlay.mp4"

# -----------------------------
# Run
# -----------------------------

create_overlay(
    video=VIDEO_PATH,
    csv=CSV_PATH,
    output=OUTPUT_PATH
)